# SWYNEX Task 2: Machine Learning Model

This notebook trains and evaluates two classification models on the Titanic dataset: Logistic Regression and Decision Tree.

**Objective:** Compare model performance using accuracy, precision, recall, and F1-score.

In [ ]:
# SWYNEX Task 2 - Machine Learning Model

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load Titanic dataset
import seaborn as sns
df = sns.load_dataset("titanic")

# Select features and target
df = df[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "survived"]]
df = df.dropna(subset=["survived"])

X = df.drop("survived", axis=1)
y = df["survived"]

# Separate numerical and categorical columns
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["pclass", "sex", "embarked"]

# Preprocessing pipelines
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Model 1: Logistic Regression
model1 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])
model1.fit(X_train, y_train)
pred1 = model1.predict(X_test)

# Model 2: Decision Tree
model2 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=5, random_state=42))
])
model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)

# Evaluate both models
acc1 = accuracy_score(y_test, pred1)
acc2 = accuracy_score(y_test, pred2)

print("===== MODEL EVALUATION =====")
print(f"Logistic Regression Accuracy: {acc1 * 100:.2f}%")
print(f"Decision Tree Accuracy: {acc2 * 100:.2f}%")

print("\\n===== LOGISTIC REGRESSION REPORT =====")
print(classification_report(y_test, pred1))

print("\\n===== DECISION TREE REPORT =====")
print(classification_report(y_test, pred2))

# Compare model performance
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Accuracy (%)": [round(acc1 * 100, 2), round(acc2 * 100, 2)]
})

print("\\n===== MODEL COMPARISON =====")
display(results)

best_model = "Logistic Regression" if acc1 >= acc2 else "Decision Tree"
print("\\nBetter performing model:", best_model)

# Conclusion
print("\\n===== CONCLUSION =====")
print(
    f"Logistic Regression performed better with {acc1 * 100:.2f}% accuracy, "
    f"compared to Decision Tree with {acc2 * 100:.2f}% accuracy."
)
